In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

# Def

In [2]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
def tokenize_function(examples):
    return tokenizer(examples["content"], padding="max_length", truncation=True)

In [3]:
metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Loading dataset and splitting dataset

In [4]:
data = pd.read_csv('https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv', encoding='latin-1')
data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
data.columns = ['label', 'content']
data["label"] = data["label"].map({"spam": 1, "ham": 0})
data.head()

,label,content
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
train, test = train_test_split(data, test_size=0.3, random_state=42)

In [6]:
train_dataset = Dataset.from_pandas(train, preserve_index=False)
test_dataset = Dataset.from_pandas(test, preserve_index=False)
ds = DatasetDict({'train': train_dataset, 'test': test_dataset})

In [7]:
tokenized_datasets = ds.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

Map:   0%|          | 0/3900 [00:00<?, ? examples/s]

Map:   0%|          | 0/1672 [00:00<?, ? examples/s]

# Training

In [8]:
model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Ega22\AppData\Local\Temp\ipykernel_19616\4212309393.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,0.000700,0.037927,0.972477
2,0.097700,0.036960,0.979405
3,0.000100,0.038042,0.979310


TrainOutput(global_step=1464, training_loss=0.03175595491854035, metrics={'train_runtime': 723.4556, 'train_samples_per_second': 16.172, 'train_steps_per_second': 2.024, 'total_flos': 3078399347712000.0, 'train_loss': 0.03175595491854035, 'epoch': 3.0})

# Metrics

In [11]:
metrics = trainer.evaluate()
print("Evaluation metrics:", metrics)

Evaluation metrics: {'eval_loss': 0.0380421057343483, 'eval_f1': 0.9793103448275862, 'eval_runtime': 27.2041, 'eval_samples_per_second': 61.461, 'eval_steps_per_second': 7.683, 'epoch': 3.0}


In [15]:
result = trainer.evaluate()
print("=" * 40)
print(f"Validation Loss: {result['eval_loss']}")
print(f"F1-score: {result['eval_f1']}")
print("=" * 40)

Validation Loss: 0.0380421057343483
F1-score: 0.9793103448275862


# Examples

In [18]:
example_text = data["content"].iloc[0]
tokenized_example = tokenizer(example_text, padding="max_length", truncation=True)

print("Original Text:")
print(example_text)

print("\nTokens:")
tokens = tokenizer.convert_ids_to_tokens(tokenized_example["input_ids"])
print(tokens)

print("\nToken IDs:")
print(tokenized_example["input_ids"])

Original Text:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Tokens:
['[CLS]', 'Go', 'until', 'j', '##uron', '##g', 'point', ',', 'crazy', '.', '.', 'Ava', '##ila', '##ble', 'only', 'in', 'bug', '##is', 'n', 'great', 'world', 'la', 'e', 'b', '##uff', '##et', '.', '.', '.', 'C', '##ine', 'there', 'got', 'am', '##ore', 'wa', '##t', '.', '.', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '